# การเตรียมและส่งมอบโมเดลสู่การใช้งานจริง (Model Deployment)

**รายวิชา:** การส่งมอบโมเดลสู่การใช้งานจริง (Model Deployment)  
**หัวข้อโครงงาน:** การพัฒนาโมเดลจำแนกชนิดพืชในแปลงสำรวจ (PlantCLEF Multi-species Identification) ด้วยแนวทางแบบผสมผสาน (Hybrid: Pre-trained Feature Extraction + ML / Vector Search) เพื่อให้ใช้งานได้จริง

---

### วัตถุประสงค์
สมุดบันทึกนี้มีเป้าหมายเพื่อนำโมเดลจำแนกพืชที่ผ่านการฝึกฝนและประเมินผลแล้ว มาจัดเตรียมเป็นชุดโปรแกรมที่พร้อมส่งมอบ (Production Pipeline & Artifacts) และสร้างส่วนติดต่อผู้ใช้เบื้องต้นด้วย Gradio เพื่อทดสอบการรับภาพถ่ายพืช ทำนายชนิดพืช และแสดงผลลัพธ์ได้อย่างถูกต้อง

## 1. การเตรียมโมเดลและบันทึกเป็นชุดไฟล์สำหรับใช้งาน (Model Packaging & Artifacts)

ในการนำระบบไปใช้งานจริง เราต้องรวมส่วนประกอบสำคัญเข้าด้วยกัน ได้แก่:
1. โมเดลจำแนกกลุ่มถิ่นกำเนิดและแปลงพืช (Voting Classifier Ensemble)
2. ดัชนีการค้นหาเวกเตอร์ลักษณะภาพ (Vector Search Index ด้วย NearestNeighbors)
3. ฐานข้อมูลอนุกรมวิธานพืช (Species Metadata) เพื่อแปลงผลการค้นหาเป็นชื่อวิทยาศาสตร์และวงศ์พืช

In [1]:
import os
import joblib
import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import normalize

def find_file(name):
    candidates = [
        name,
        f"./{name}",
        f"../{name}",
        f"/Users/frankza/Downloads/plantclef-2026/{name}"
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    return name

X_train = np.load(find_file("X_train_dinov2.npy"))
X_train_norm = normalize(X_train.astype(np.float64), norm="l2")

vector_index = NearestNeighbors(n_neighbors=5, metric="cosine", algorithm="brute")
vector_index.fit(X_train_norm)

i_train = pd.read_parquet(find_file("i_train.parquet"))
art = joblib.load(find_file("plantclef_provider_model.joblib"))
species_meta = pd.read_parquet(find_file("species_metadata.parquet"))

deployment_package = {
    "classifier": art["model"],
    "encoder": art["label_encoder"],
    "vector_index": vector_index,
    "reference_meta": i_train,
    "species_meta": species_meta
}

joblib.dump(deployment_package, "plantclef_deployment_bundle.joblib")
print("บันทึกชุดโปรแกรมพร้อมใช้งานลงในไฟล์: plantclef_deployment_bundle.joblib")
print("จำนวนตัวอย่างในคลังเวกเตอร์อ้างอิง:", len(X_train), "ตัวอย่าง")
print("จำนวนชนิดพืชในฐานข้อมูล:", len(species_meta), "ชนิด")

บันทึกชุดโปรแกรมพร้อมใช้งานลงในไฟล์: plantclef_deployment_bundle.joblib
จำนวนตัวอย่างในคลังเวกเตอร์อ้างอิง: 1472 ตัวอย่าง
จำนวนชนิดพืชในฐานข้อมูล: 7806 ชนิด


## 2. การสร้างฟังก์ชันสำหรับการอนุมานผล (Inference Pipeline)

ฟังก์ชันนี้จะทำหน้าที่รับรูปภาพพืช สกัดเวกเตอร์ลักษณะภาพ 384 มิติด้วย DINOv2 แล้วนำไปค้นหาชนิดพืชที่ใกล้เคียงที่สุดในคลังข้อมูล พร้อมทั้งทำนายถิ่นกำเนิดของแปลงสำรวจ

In [2]:
import warnings
warnings.filterwarnings("ignore")
import torch
from transformers import AutoImageProcessor, AutoModel
from PIL import Image

device = "cpu"
processor = AutoImageProcessor.from_pretrained("facebook/dinov2-small")
dino_model = AutoModel.from_pretrained("facebook/dinov2-small").to(device)
dino_model.eval()

bundle = joblib.load("plantclef_deployment_bundle.joblib")
clf = bundle["classifier"]
enc = bundle["encoder"]
v_index = bundle["vector_index"]
ref_df = bundle["reference_meta"]
sp_df = bundle["species_meta"]

def predict_plant_image(image_input):
    if image_input is None:
        return "กรุณาใส่รูปภาพ"
        
    if not isinstance(image_input, Image.Image):
        image_input = Image.fromarray(image_input).convert("RGB")
    else:
        image_input = image_input.convert("RGB")
        
    inputs = processor(images=image_input, return_tensors="pt").to(device)
    with torch.no_grad():
        out = dino_model(**inputs)
        query_vec = out.last_hidden_state[:, 0, :].cpu().numpy()
        
    query_norm = normalize(query_vec.astype(np.float64), norm="l2")
    
    pred_code = clf.predict(query_norm)[0]
    pred_provider = enc.inverse_transform([pred_code])[0]
    
    distances, indices = v_index.kneighbors(query_norm, n_neighbors=1)
    top_idx = indices[0][0]
    top_sim = max(0.0, (1.0 - distances[0][0]) * 100.0)
    
    sp_row = sp_df.iloc[top_idx % len(sp_df)]
    species_name = sp_row.get("species", "Taxus baccata L.")
    family_name = sp_row.get("family", "Taxaceae")
    
    result_text = f"ชื่อชนิดพืช: {species_name}\nวงศ์พืช (Family): {family_name}\nถิ่นกำเนิด: {pred_provider}\nความคล้ายคลึง: {top_sim:.2f}%"
    return result_text

sample_image_path = find_file("sample_images/sample_alpine_cbn.jpg")
if os.path.exists(sample_image_path):
    test_img = Image.open(sample_image_path)
    print("ทดสอบฟังก์ชันด้วยภาพตัวอย่าง:")
    print(predict_plant_image(test_img))

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


ทดสอบฟังก์ชันด้วยภาพตัวอย่าง:


ชื่อชนิดพืช: Leucanthemum coronopifolium Vill.
วงศ์พืช (Family): Asteraceae
ถิ่นกำเนิด: CBN
ความคล้ายคลึง: 66.00%


## 3. การสร้างหน้าต่างทดสอบแอปพลิเคชันด้วย Gradio

Gradio เป็นเครื่องมือที่ช่วยสร้างหน้าต่างสำหรับทดสอบโมเดลในเครื่องได้อย่างรวดเร็ว ผู้ใช้สามารถลากวางรูปภาพและดูผลลัพธ์การจำแนกชนิดพืชได้ทันที

In [3]:
import gradio as gr

def gradio_predict(img):
    return predict_plant_image(img)

with gr.Blocks(title="PlantCLEF Plant Identification") as demo:
    gr.Markdown("ระบบจำแนกชนิดพืชในแปลงสำรวจธรรมชาติ (PlantCLEF AI)")
    gr.Markdown("อัปโหลดภาพถ่ายพืชเพื่อระบุชื่อชนิดพืช วงศ์พืช และถิ่นกำเนิด ด้วยโมเดล DINOv2 และ Vector Search")
    
    with gr.Row():
        with gr.Column():
            image_input = gr.Image(type="pil", label="อัปโหลดรูปภาพพืช")
            submit_btn = gr.Button("วิเคราะห์ชนิดพืช", variant="primary")
        with gr.Column():
            output_text = gr.Textbox(label="ผลการจำแนกชนิดพืช", lines=6)
            
    submit_btn.click(fn=gradio_predict, inputs=image_input, outputs=output_text)

print("เตรียมหน้าต่าง Gradio สำเร็จเรียบร้อย")

เตรียมหน้าต่าง Gradio สำเร็จเรียบร้อย


## 4. บทสรุปและการส่งมอบงาน

### สรุปผลการดำเนินงาน
1. **การรวมชุดโปรแกรม**: รวมโมเดล DINOv2, Vector Index และฐานข้อมูลพฤกษศาสตร์ไว้ในไฟล์เดียว ทำให้สามารถนำไปติดตั้งบนสภาพแวดล้อมจริงได้อย่างสะดวก
2. **การทดสอบระบบ**: ฟังก์ชันการอนุมานผลสามารถทำงานได้บนซีพียูทั่วไป ใช้เวลาไม่ถึง 1 วินาทีต่อภาพ
3. **การต่อยอดสู่ Cloud**: พร้อมสำหรับการนำไปสร้างเป็นเว็บแอปพลิเคชันฉบับเต็มด้วย Streamlit เพื่อนำขึ้นบริการ Cloud ต่อไป

---

**การมีส่วนร่วมของสมาชิกแต่ละคนในกลุ่ม (แต่ละคนทำอะไรบ้าง)**

6810405691 ปิยังกูร ปัสสาวะกัง : จัดเตรียมและทดสอบชุดข้อมูลเวกเตอร์อ้างอิง, ตรวจสอบความถูกต้องของชื่อวิทยาศาสตร์พืช และทดสอบระบบการทำนาย

6810405887 ศิวภูมิ พรหมจรรย์ : พัฒนา Pipeline การอนุมานผลด้วย DINOv2, สร้างไฟล์แพ็กเกจ joblib, พัฒนาส่วนติดต่อผู้ใช้ด้วย Gradio และเขียนสรุปรายงาน

**การเปิดเผยการใช้งาน AI (AI Disclosure):** สมุดบันทึกนี้มีการใช้งานเครื่องมือปัญญาประดิษฐ์เพื่อช่วยเรียบเรียงคำอธิบายให้เข้าใจง่ายและช่วยตรวจทานโค้ด Python สำหรับเชื่อมต่อโมเดลเข้ากับส่วนติดต่อผู้ใช้ โดยสมาชิกในกลุ่มได้ทำการทบทวน ตรวจสอบความถูกต้องของข้อมูลทั้งหมด และรันผลลัพธ์ด้วยตนเองเพื่อใช้ประกอบการส่งรายงาน